# 01 ? Data understanding and quality audit

**Question:** What do the source files contain, and can their identifiers support a reliable household-level Customer 360?

This notebook calculates results directly from the original CSVs. SQL is displayed next to its output. Shared checks come from `src/retail_customer_360/audit.py`, so the notebook and automated audit use the same definitions. No precomputed report is loaded and no source rows are changed.

Run all cells from top to bottom using the project `.venv` Python interpreter. Full-file profiling includes 36+ million promotional records and can take several minutes. Outputs are retained for GitHub readers.

**Grain:** a transaction row is a product-line observation; a basket is a shopping receipt; a household is the observed customer entity. These are different counts. Source `DAY` values are relative day numbers, not verified calendar dates.

**Step 1: Prepare imports, imports, imports.**

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

ROOT = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "src/retail_customer_360/audit.py").is_file()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Open the notebook inside the retail-customer-360 repository.")
sys.path.insert(0, str(ROOT / "src"))


**Step 2: Import the required libraries.**

In [2]:
from retail_customer_360.audit import (
    connect_sources,
    EXPECTED_FILES,
    QUERIES,
    profiling_sql,
    profile_file,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)


**Step 3: Define the run_sql helper.**

In [3]:
def run_sql(query):
    display(Markdown("```sql\n" + query.strip() + "\n```"))
    result = con.execute(query).fetchdf()
    display(result)
    return result


**Step 4: Continue the calculation.**

In [4]:
print("Project:", ROOT.name)
print("Python:", sys.version.split()[0])


Project: retail-customer-360
Python: 3.12.15


## 1. Files and inferred schema

Inventory the original files, then expose them as read-only DuckDB views. Types are inferred for exploration; production schemas still need explicit validation. The connection setup scans all files to infer types.

**Step 5: Prepare con.**

In [5]:
display(
    pd.DataFrame([
        {
            "file": name + ".csv",
            "size_MiB": round(
                (ROOT / "data/raw" / (name + ".csv")).stat().st_size / 1024**2, 2
            ),
        }
        for name in EXPECTED_FILES
    ])
)
con = connect_sources(ROOT)


,file,size_MiB
0,campaign_desc.csv,0.00
1,campaign_table.csv,0.09
2,causal_data.csv,663.62
3,coupon.csv,2.69
4,coupon_redempt.csv,0.05
5,hh_demographic.csv,0.04
6,product.csv,6.13
7,transaction_data.csv,135.18


**Step 6: Prepare schema.**

In [6]:
schema = pd.concat(
    [
        con.execute(f"DESCRIBE {name}").fetchdf().assign(source_table=name)
        for name in EXPECTED_FILES
    ],
    ignore_index=True,
)
display(schema[["source_table", "column_name", "column_type"]])


,source_table,column_name,column_type
0,campaign_desc,DESCRIPTION,VARCHAR
1,campaign_desc,CAMPAIGN,BIGINT
2,campaign_desc,START_DAY,BIGINT
3,campaign_desc,END_DAY,BIGINT
4,campaign_table,DESCRIPTION,VARCHAR
5,campaign_table,household_key,BIGINT
6,campaign_table,CAMPAIGN,BIGINT
7,causal_data,PRODUCT_ID,BIGINT
8,causal_data,STORE_ID,BIGINT
9,causal_data,WEEK_NO,BIGINT


## 2. Row counts, missing values, and whitespace

`COUNT(*)` counts source rows. `IS NULL` detects SQL missing values. `TRIM(...) = ''` separately catches whitespace-only values: these would be missed by a NULL-only check. The query below demonstrates the exact shared profiling logic on the product table. We then apply it to every file.

Blank strings and NULLs are reported separately and are not automatically replaced.

**Step 7: Prepare profiles, file counts.**

In [7]:
display(Markdown("```sql\n" + profiling_sql(con, "product") + "\n```"))
profiles = {}
for name in EXPECTED_FILES:
    profiles[name] = profile_file(con, name, ROOT)
    print(f"{name}: {profiles[name]['rows']:,} rows")
file_counts = pd.DataFrame([
    {
        "table": name,
        "rows": item["rows"],
        "null_cells": sum(c["nulls"] for c in item["columns"]),
        "blank_string_cells": sum(c["blank_strings"] for c in item["columns"]),
    }
    for name, item in profiles.items()
])
display(file_counts)


```sql
SELECT
    count(*) AS row_count,
    count(*) FILTER (WHERE "PRODUCT_ID" IS NULL) AS null_0,
    count(*) FILTER (WHERE "PRODUCT_ID" IS NOT NULL AND trim(cast("PRODUCT_ID" AS VARCHAR)) = '') AS blank_0,
    count(*) FILTER (WHERE "MANUFACTURER" IS NULL) AS null_1,
    count(*) FILTER (WHERE "MANUFACTURER" IS NOT NULL AND trim(cast("MANUFACTURER" AS VARCHAR)) = '') AS blank_1,
    count(*) FILTER (WHERE "DEPARTMENT" IS NULL) AS null_2,
    count(*) FILTER (WHERE "DEPARTMENT" IS NOT NULL AND trim(cast("DEPARTMENT" AS VARCHAR)) = '') AS blank_2,
    count(*) FILTER (WHERE "BRAND" IS NULL) AS null_3,
    count(*) FILTER (WHERE "BRAND" IS NOT NULL AND trim(cast("BRAND" AS VARCHAR)) = '') AS blank_3,
    count(*) FILTER (WHERE "COMMODITY_DESC" IS NULL) AS null_4,
    count(*) FILTER (WHERE "COMMODITY_DESC" IS NOT NULL AND trim(cast("COMMODITY_DESC" AS VARCHAR)) = '') AS blank_4,
    count(*) FILTER (WHERE "SUB_COMMODITY_DESC" IS NULL) AS null_5,
    count(*) FILTER (WHERE "SUB_COMMODITY_DESC" IS NOT NULL AND trim(cast("SUB_COMMODITY_DESC" AS VARCHAR)) = '') AS blank_5,
    count(*) FILTER (WHERE "CURR_SIZE_OF_PRODUCT" IS NULL) AS null_6,
    count(*) FILTER (WHERE "CURR_SIZE_OF_PRODUCT" IS NOT NULL AND trim(cast("CURR_SIZE_OF_PRODUCT" AS VARCHAR)) = '') AS blank_6
FROM product
```

campaign_desc: 30 rows
campaign_table: 7,208 rows


causal_data: 36,786,524 rows


coupon: 124,548 rows
coupon_redempt: 2,318 rows
hh_demographic: 801 rows


product: 92,353 rows


transaction_data: 2,595,732 rows


,table,rows,null_cells,blank_string_cells
0,campaign_desc,30,0,0
1,campaign_table,7208,0,0
2,causal_data,36786524,0,0
3,coupon,124548,0,0
4,coupon_redempt,2318,0,0
5,hh_demographic,801,0,0
6,product,92353,0,30652
7,transaction_data,2595732,0,0


In [8]:
display(pd.DataFrame([{"table": k, "rows": v["rows"]} for k, v in profiles.items()]))


,table,rows
0,campaign_desc,30
1,campaign_table,7208
2,causal_data,36786524
3,coupon,124548
4,coupon_redempt,2318
5,hh_demographic,801
6,product,92353
7,transaction_data,2595732


**Step 8: Prepare missingness.**

In [9]:
missingness = pd.DataFrame([
    {
        "table": name,
        "column": col["name"],
        "type": col["inferred_type"],
        "nulls": col["nulls"],
        "blank_strings": col["blank_strings"],
        "missing_or_blank_pct": (
            100 * (col["nulls"] + col["blank_strings"]) / item["rows"]
        ),
    }
    for name, item in profiles.items()
    for col in item["columns"]
])
display(missingness)
display(missingness.query("nulls > 0 or blank_strings > 0"))


,table,column,type,nulls,blank_strings,missing_or_blank_pct
0,campaign_desc,DESCRIPTION,VARCHAR,0,0,0.000000
1,campaign_desc,CAMPAIGN,BIGINT,0,0,0.000000
2,campaign_desc,START_DAY,BIGINT,0,0,0.000000
3,campaign_desc,END_DAY,BIGINT,0,0,0.000000
4,campaign_table,DESCRIPTION,VARCHAR,0,0,0.000000
5,campaign_table,household_key,BIGINT,0,0,0.000000
6,campaign_table,CAMPAIGN,BIGINT,0,0,0.000000
7,causal_data,PRODUCT_ID,BIGINT,0,0,0.000000
8,causal_data,STORE_ID,BIGINT,0,0,0.000000
9,causal_data,WEEK_NO,BIGINT,0,0,0.000000


,table,column,type,nulls,blank_strings,missing_or_blank_pct
29,product,DEPARTMENT,VARCHAR,0,15,0.016242
31,product,COMMODITY_DESC,VARCHAR,0,15,0.016242
32,product,SUB_COMMODITY_DESC,VARCHAR,0,15,0.016242
33,product,CURR_SIZE_OF_PRODUCT,VARCHAR,0,30607,33.141316


**Interpretation:** source parsing may recognise additional missing-value conventions; zero SQL NULLs would not prove every field is semantically complete. Whitespace-only product attributes need an explicit cleaning rule later.

## 3. Transactions, baskets, households, and ranges

`COUNT(*)` gives product-line observations. `COUNT(DISTINCT BASKET_ID)` gives observed receipts. `COUNT(DISTINCT household_key)` gives purchasing households. No eligibility exclusions have been applied.

The shared overview includes source day coverage, distinct products/stores, source sales totals, and anomaly counts. The sales sum is a sum of the source field, not a verified gross/net revenue or profit definition.

**Step 9: Calculate and inspect query results.**

In [10]:
counts = run_sql(
    """SELECT count(*) AS transaction_rows, count(DISTINCT basket_id) AS baskets,
       count(DISTINCT household_key) AS households
FROM transaction_data"""
)
overview = run_sql(QUERIES["transaction_overview"])


```sql
SELECT count(*) AS transaction_rows, count(DISTINCT basket_id) AS baskets,
       count(DISTINCT household_key) AS households
FROM transaction_data
```

,transaction_rows,baskets,households
0,2595732,276484,2500


```sql
SELECT min(day) AS first_day, max(day) AS last_day,
 count(DISTINCT household_key) AS households, count(DISTINCT basket_id) AS baskets,
 count(DISTINCT product_id) AS products, count(DISTINCT store_id) AS stores,
 sum(sales_value) AS source_sales_value_sum,
 count(*) FILTER (WHERE quantity < 0) AS negative_quantity_rows,
 count(*) FILTER (WHERE quantity = 0) AS zero_quantity_rows,
 count(*) FILTER (WHERE sales_value < 0) AS negative_sales_rows,
 count(*) FILTER (WHERE sales_value = 0) AS zero_sales_rows,
 count(*) FILTER (WHERE retail_disc > 0 OR coupon_disc > 0 OR coupon_match_disc > 0) AS positive_discount_rows
 FROM transaction_data
```

,first_day,last_day,households,baskets,products,stores,source_sales_value_sum,negative_quantity_rows,zero_quantity_rows,negative_sales_rows,zero_sales_rows,positive_discount_rows
0,1,711,2500,276484,92339,582,8.057463e+06,0,14466,0,18850,36


**Step 10: Calculate and inspect query results.**

In [11]:
ranges = run_sql(
    "SELECT min(quantity) AS min_quantity, max(quantity) AS "
    "max_quantity,\n    min(sales_value) AS min_sales_value, "
    "max(sales_value) AS max_sales_value,\n    min(retail_disc) "
    "AS min_retail_discount, max(retail_disc) AS "
    "max_retail_discount,\n    min(coupon_disc) AS "
    "min_coupon_discount, max(coupon_disc) AS "
    "max_coupon_discount,\n    min(coupon_match_disc) AS "
    "min_coupon_match, max(coupon_match_disc) AS "
    "max_coupon_match,\n    min(week_no) AS first_week, "
    "max(week_no) AS last_week,\n    min(trans_time) AS "
    "min_source_time, max(trans_time) AS max_source_time\nFROM "
    "transaction_data"
)


```sql
SELECT min(quantity) AS min_quantity, max(quantity) AS max_quantity,
    min(sales_value) AS min_sales_value, max(sales_value) AS max_sales_value,
    min(retail_disc) AS min_retail_discount, max(retail_disc) AS max_retail_discount,
    min(coupon_disc) AS min_coupon_discount, max(coupon_disc) AS max_coupon_discount,
    min(coupon_match_disc) AS min_coupon_match, max(coupon_match_disc) AS max_coupon_match,
    min(week_no) AS first_week, max(week_no) AS last_week,
    min(trans_time) AS min_source_time, max(trans_time) AS max_source_time
FROM transaction_data
```

,min_quantity,max_quantity,min_sales_value,max_sales_value,min_retail_discount,max_retail_discount,min_coupon_discount,max_coupon_discount,min_coupon_match,max_coupon_match,first_week,last_week,min_source_time,max_source_time
0,0,89638,0.0,840.0,-180.0,3.99,-55.93,0.0,-7.7,0.0,1,102,0000,2359


## 4. See how lines form baskets and household histories

The following examples use the smallest observed basket identifier and a household with at least two baskets. They are deterministic illustrations, not representative samples. Basket totals count source lines and sum their source sales values; they do not redefine sales semantics.

**Step 11: Calculate and inspect query results.**

In [12]:
basket_lines = run_sql(
    """SELECT household_key, basket_id, day, store_id, product_id, quantity,
       sales_value
FROM transaction_data
WHERE basket_id =
    (SELECT min(basket_id)
     FROM transaction_data)
ORDER BY product_id, quantity, sales_value"""
)
basket_totals = run_sql("""SELECT basket_id, min(household_key) AS household_key,
    min(day) AS source_day, count(*) AS line_rows,
    count(DISTINCT product_id) AS distinct_products,
    sum(sales_value) AS source_sales_sum
FROM transaction_data
WHERE basket_id = (SELECT min(basket_id) FROM transaction_data)
GROUP BY basket_id""")


```sql
SELECT household_key, basket_id, day, store_id, product_id, quantity,
       sales_value
FROM transaction_data
WHERE basket_id =
    (SELECT min(basket_id)
     FROM transaction_data)
ORDER BY product_id, quantity, sales_value
```

,household_key,BASKET_ID,DAY,STORE_ID,PRODUCT_ID,QUANTITY,SALES_VALUE
0,2375,26984851472,1,364,1004906,1,1.39
1,2375,26984851472,1,364,1033142,1,0.82
2,2375,26984851472,1,364,1036325,1,0.99
3,2375,26984851472,1,364,1082185,1,1.21
4,2375,26984851472,1,364,8160430,1,1.50


```sql
SELECT basket_id, min(household_key) AS household_key,
    min(day) AS source_day, count(*) AS line_rows,
    count(DISTINCT product_id) AS distinct_products,
    sum(sales_value) AS source_sales_sum
FROM transaction_data
WHERE basket_id = (SELECT min(basket_id) FROM transaction_data)
GROUP BY basket_id
```

,BASKET_ID,household_key,source_day,line_rows,distinct_products,source_sales_sum
0,26984851472,2375,1,5,5,5.91


**Step 12: Calculate and inspect query results.**

In [13]:
history = run_sql("""WITH example AS
  (SELECT household_key
   FROM transaction_data
   GROUP BY household_key
   HAVING count(DISTINCT basket_id) >= 2
   ORDER BY household_key
   LIMIT 1)
SELECT household_key, basket_id, day, store_id, count(*) AS line_rows,
       sum(sales_value) AS source_sales_sum
FROM transaction_data
JOIN example USING(household_key)
GROUP BY household_key, basket_id, day, store_id
ORDER BY day, basket_id
LIMIT 10""")


```sql
WITH example AS
  (SELECT household_key
   FROM transaction_data
   GROUP BY household_key
   HAVING count(DISTINCT basket_id) >= 2
   ORDER BY household_key
   LIMIT 1)
SELECT household_key, basket_id, day, store_id, count(*) AS line_rows,
       sum(sales_value) AS source_sales_sum
FROM transaction_data
JOIN example USING(household_key)
GROUP BY household_key, basket_id, day, store_id
ORDER BY day, basket_id
LIMIT 10
```

,household_key,BASKET_ID,DAY,STORE_ID,line_rows,source_sales_sum
0,1,27601281299,51,436,30,78.66
1,1,27774192959,67,436,12,41.10
2,1,28024266849,88,436,12,26.90
3,1,28106322445,94,436,23,63.43
4,1,28235481967,101,436,17,53.45
5,1,28282581446,108,436,10,26.76
6,1,28318295376,111,31401,9,23.55
7,1,28601601153,128,436,39,110.34
8,1,28744672720,137,436,36,87.44
9,1,28853646373,146,436,29,73.32


## 5. Duplicate rows and candidate keys

Exact full-row duplicates and duplicated dimension keys answer different questions. A repeated basket ID is expected in line-level data. A repeated basket-product combination also needs investigation before treating it as an error.

Each result below executes the shared audit query. Zero duplicate keys does not replace a separate missing-key check (see section 2). Coupon mapping and promotional exposure uniqueness are outside this initial key audit.

In [14]:
key_results = {}
for check in [
    "transaction_exact_duplicates",
    "product_key_duplicates",
    "demographic_key_duplicates",
    "campaign_key_duplicates",
    "campaign_assignment_duplicates",
]:
    display(Markdown("**" + check + "**"))
    key_results[check] = run_sql(QUERIES[check])
repeated_pairs = run_sql("""SELECT basket_id, product_id, count(*) AS source_rows
FROM transaction_data
GROUP BY basket_id, product_id
HAVING count(*) > 1
ORDER BY source_rows DESC, basket_id, product_id
LIMIT 10""")


**transaction_exact_duplicates**

```sql
SELECT count(*) AS duplicate_groups, coalesce(sum(n-1),
                                       0) AS excess_rows
FROM
  (SELECT *, count(*) AS n
   FROM transaction_data
   GROUP BY ALL
   HAVING count(*) > 1)
```

,duplicate_groups,excess_rows
0,0,0.0


**product_key_duplicates**

```sql
SELECT count(*) AS duplicate_keys
FROM
  (SELECT product_id
   FROM product
   GROUP BY product_id
   HAVING count(*)>1)
```

,duplicate_keys
0,0


... (7 outputs left out)

```sql
SELECT count(*) AS duplicate_keys
FROM
  (SELECT household_key,campaign
   FROM campaign_table
   GROUP BY ALL
   HAVING count(*)>1)
```

,duplicate_keys
0,0


```sql
SELECT basket_id, product_id, count(*) AS source_rows
FROM transaction_data
GROUP BY basket_id, product_id
HAVING count(*) > 1
ORDER BY source_rows DESC, basket_id, product_id
LIMIT 10
```

,BASKET_ID,PRODUCT_ID,source_rows


## 6. Does each basket belong to one household, day, and store?

Group by basket and count distinct values of its expected receipt-level attributes. Any count greater than one is a conflict. Missing values are checked separately because `COUNT(DISTINCT ...)` ignores NULLs.

In [15]:
basket_consistency = run_sql(QUERIES["basket_consistency"])
basket_conflicts = run_sql(
    "SELECT basket_id,\n    count(DISTINCT household_key) AS "
    "household_count,\n    count(DISTINCT day) AS day_count, "
    "count(DISTINCT store_id) AS store_count\nFROM "
    "transaction_data GROUP BY basket_id\nHAVING count(DISTINCT "
    "household_key)>1 OR count(DISTINCT day)>1 OR count(DISTINCT "
    "store_id)>1\nORDER BY basket_id LIMIT 10"
)


```sql
SELECT count(*) AS inconsistent_baskets
FROM
  (SELECT basket_id
   FROM transaction_data
   GROUP BY basket_id
   HAVING count(DISTINCT household_key)>1
   OR count(DISTINCT day)>1
   OR count(DISTINCT store_id)>1)
```

,inconsistent_baskets
0,0


```sql
SELECT basket_id,
    count(DISTINCT household_key) AS household_count,
    count(DISTINCT day) AS day_count, count(DISTINCT store_id) AS store_count
FROM transaction_data GROUP BY basket_id
HAVING count(DISTINCT household_key)>1 OR count(DISTINCT day)>1 OR count(DISTINCT store_id)>1
ORDER BY basket_id LIMIT 10
```

,BASKET_ID,household_count,day_count,store_count


## 7. Products and campaign assignments without reference records

An `ANTI JOIN` keeps source rows for which no matching reference identifier exists. We count unmatched transaction rows and show examples. A clean product relationship does not establish that every other relationship is complete.

In [16]:
product_reference = run_sql(QUERIES["product_reference"])
unmatched_products = run_sql("""SELECT t.product_id, count(*) AS unmatched_rows
FROM transaction_data t ANTI
JOIN product p USING(product_id)
GROUP BY t.product_id
ORDER BY unmatched_rows DESC, t.product_id
LIMIT 10""")
campaign_reference = run_sql(QUERIES["campaign_reference"])


```sql
SELECT count(*) AS unmatched_transaction_rows
FROM transaction_data t ANTI
JOIN product p USING(product_id)
```

,unmatched_transaction_rows
0,0


```sql
SELECT t.product_id, count(*) AS unmatched_rows
FROM transaction_data t ANTI
JOIN product p USING(product_id)
GROUP BY t.product_id
ORDER BY unmatched_rows DESC, t.product_id
LIMIT 10
```

,PRODUCT_ID,unmatched_rows


```sql
SELECT count(*) AS unmatched_assignments
FROM campaign_table t ANTI
JOIN campaign_desc d USING(campaign)
```

,unmatched_assignments
0,0


## 8. Demographic coverage and join choice

Start from all purchasing households and **left join** demographics. An inner join would drop households without demographic records and change the analytical population. Missing demographics should remain explicit.

In [17]:
coverage = run_sql(QUERIES["demographic_coverage"])
coverage_summary = coverage.copy()
coverage_summary["without_demographics"] = (
    coverage_summary["purchasing_households"] - coverage_summary["with_demographics"]
)
coverage_summary["coverage_pct"] = (
    100
    * coverage_summary["with_demographics"]
    / coverage_summary["purchasing_households"]
)
display(coverage_summary)
missing_households = run_sql("""SELECT DISTINCT t.household_key
FROM transaction_data t ANTI
JOIN hh_demographic d USING(household_key)
ORDER BY t.household_key
LIMIT 10""")


```sql
SELECT count(*) AS purchasing_households, count(d.household_key) AS with_demographics FROM
 (SELECT DISTINCT household_key FROM transaction_data) t LEFT JOIN hh_demographic d USING(household_key)
```

,purchasing_households,with_demographics
0,2500,801


,purchasing_households,with_demographics,without_demographics,coverage_pct
0,2500,801,1699,32.04


```sql
SELECT DISTINCT t.household_key
FROM transaction_data t ANTI
JOIN hh_demographic d USING(household_key)
ORDER BY t.household_key
LIMIT 10
```

,household_key
0,2
1,3
2,4
3,5
4,6
5,9
6,10
7,11
8,12
9,14


## 9. Zero values and discount signs: inspect before cleaning

Counts can overlap: a zero-quantity row may also have zero sales. The examples below are diagnostic selections. Negative sales or quantities cannot automatically be interpreted as returns, and their absence cannot establish that returns were never recorded elsewhere. Positive discount values also require source documentation.

**Step 17: Calculate and inspect query results.**

In [18]:
anomalies = run_sql(
    "SELECT\n    count(*) FILTER (WHERE quantity=0) AS "
    "zero_quantity_rows,\n    count(*) FILTER (WHERE "
    "sales_value=0) AS zero_sales_rows,\n    count(*) FILTER "
    "(WHERE quantity=0 AND sales_value=0) AS "
    "both_zero_rows,\n    count(*) FILTER (WHERE quantity=0 AND "
    "sales_value<>0) AS zero_quantity_nonzero_sales_rows\nFROM "
    "transaction_data"
)
zero_examples = run_sql("""SELECT *
FROM transaction_data
WHERE quantity=0
  OR sales_value=0
ORDER BY day, basket_id, product_id, quantity, sales_value
LIMIT 10""")


```sql
SELECT
    count(*) FILTER (WHERE quantity=0) AS zero_quantity_rows,
    count(*) FILTER (WHERE sales_value=0) AS zero_sales_rows,
    count(*) FILTER (WHERE quantity=0 AND sales_value=0) AS both_zero_rows,
    count(*) FILTER (WHERE quantity=0 AND sales_value<>0) AS zero_quantity_nonzero_sales_rows
FROM transaction_data
```

,zero_quantity_rows,zero_sales_rows,both_zero_rows,zero_quantity_nonzero_sales_rows
0,14466,18850,14399,67


```sql
SELECT *
FROM transaction_data
WHERE quantity=0
  OR sales_value=0
ORDER BY day, basket_id, product_id, quantity, sales_value
LIMIT 10
```

,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
0,744,26985165432,1,5978648,0,0.0,31582,0.00,1119,1,0.0,0.0
1,1287,26985336468,1,5978648,0,0.0,304,0.00,1351,1,0.0,0.0
2,2305,26996870743,2,5978656,0,0.0,414,0.00,1300,1,-1.0,0.0
3,271,26997082949,2,5978656,0,0.0,329,0.00,1719,1,-2.0,0.0
4,315,27008952267,3,957951,0,0.0,327,0.00,1707,1,0.0,0.0
5,2110,27009082349,3,837963,1,0.0,375,0.00,1547,1,0.0,0.0
6,568,27021090189,4,951412,1,0.0,446,-1.48,0035,1,0.0,0.0
7,14,27021203242,4,6463658,0,0.0,319,0.00,1704,1,0.0,0.0
8,2110,27021248054,4,850115,1,0.0,375,-3.29,1705,1,0.0,0.0
9,2110,27021248054,4,976872,1,0.0,375,-2.49,1705,1,0.0,0.0


**Step 18: Calculate and inspect query results.**

In [19]:
discount_examples = run_sql(
    """SELECT household_key, basket_id, day, product_id, sales_value,
       retail_disc, coupon_disc, coupon_match_disc
FROM transaction_data
WHERE retail_disc>0
  OR coupon_disc>0
  OR coupon_match_disc>0
ORDER BY day, basket_id, product_id
LIMIT 10"""
)


```sql
SELECT household_key, basket_id, day, product_id, sales_value,
       retail_disc, coupon_disc, coupon_match_disc
FROM transaction_data
WHERE retail_disc>0
  OR coupon_disc>0
  OR coupon_match_disc>0
ORDER BY day, basket_id, product_id
LIMIT 10
```

,household_key,BASKET_ID,DAY,PRODUCT_ID,SALES_VALUE,RETAIL_DISC,COUPON_DISC,COUPON_MATCH_DISC
0,1306,31624096856,310,990941,1.000000e-02,1.000000e-02,0.0,0.0
1,1306,31735306143,318,7409789,0.000000e+00,2.220446e-16,0.0,0.0
2,1321,31803641901,323,1051069,0.000000e+00,5.551115e-17,0.0,0.0
3,1321,31803641901,323,1055863,0.000000e+00,5.551115e-17,0.0,0.0
4,37,31944616637,334,8090539,0.000000e+00,2.220446e-16,0.0,0.0
5,2241,32173242560,353,995242,0.000000e+00,2.220446e-16,0.0,0.0
6,923,32446036423,369,1047226,0.000000e+00,1.110223e-16,0.0,0.0
7,1535,32478752229,370,999270,7.980000e+00,3.990000e+00,0.0,0.0
8,2491,32478901200,370,13945244,4.440892e-16,1.110223e-16,0.0,0.0
9,623,32704598135,384,5995423,5.100000e-01,2.600000e-01,0.0,0.0


**Step 19: Calculate and inspect query results.**

In [20]:
blank_products = run_sql(
    """SELECT product_id, department, commodity_desc, sub_commodity_desc,
       curr_size_of_product
FROM product
WHERE trim(department)=''
  OR trim(commodity_desc)=''
ORDER BY product_id
LIMIT 10"""
)


```sql
SELECT product_id, department, commodity_desc, sub_commodity_desc,
       curr_size_of_product
FROM product
WHERE trim(department)=''
  OR trim(commodity_desc)=''
ORDER BY product_id
LIMIT 10
```

,PRODUCT_ID,DEPARTMENT,COMMODITY_DESC,SUB_COMMODITY_DESC,CURR_SIZE_OF_PRODUCT
0,5126087,,,,
1,5126088,,,,
2,5126106,,,,
3,5126107,,,,
4,5977100,,,,
5,5978648,,,,
6,5978649,,,,
7,5978650,,,,
8,5978656,,,,
9,5978657,,,,


## 10. Campaign coverage and complete future labels

Compare campaign periods with observed transaction coverage. Campaigns ending beyond the observation horizon have incomplete post-campaign outcomes. For a label defined as no purchase in `(cutoff, cutoff + 30]`, require `cutoff + 30 <= last observed source day`. This is a coverage condition, not a chosen model split.

In [21]:
campaign_periods = run_sql(QUERIES["campaign_periods"])
coverage_horizon = run_sql("""SELECT max(day) AS last_observed_day,
       max(day)-30 AS latest_cutoff_with_full_30_day_followup
FROM transaction_data""")
late_campaigns = run_sql("""SELECT campaign, description, start_day, end_day
FROM campaign_desc
WHERE end_day >
    (SELECT max(day)
     FROM transaction_data)
ORDER BY end_day, campaign""")


```sql
SELECT min(start_day) AS first_start_day, max(end_day) AS last_end_day,
       count(*) FILTER (
                        WHERE end_day < start_day) AS invalid_periods
FROM campaign_desc
```

,first_start_day,last_end_day,invalid_periods
0,224,719,0


```sql
SELECT max(day) AS last_observed_day,
       max(day)-30 AS latest_cutoff_with_full_30_day_followup
FROM transaction_data
```

,last_observed_day,latest_cutoff_with_full_30_day_followup
0,711,681


```sql
SELECT campaign, description, start_day, end_day
FROM campaign_desc
WHERE end_day >
    (SELECT max(day)
     FROM transaction_data)
ORDER BY end_day, campaign
```

,CAMPAIGN,DESCRIPTION,START_DAY,END_DAY
0,24,TypeB,659,719


## 11. Findings calculated from these cells

The text below is generated from the executed outputs, rather than copied from the earlier Markdown report.

**Step 21: Prepare n, c, a.**

In [22]:
n = counts.iloc[0]
c = coverage_summary.iloc[0]
a = anomalies.iloc[0]
h = coverage_horizon.iloc[0]


**Step 22: Prepare final checks.**

In [23]:
final_checks = {
    "Transaction rows": int(n.transaction_rows),
    "Baskets": int(n.baskets),
    "Households": int(n.households),
    "Households with demographics": int(c.with_demographics),
    "Demographic coverage (%)": float(c.coverage_pct),
    "Zero quantity rows": int(a.zero_quantity_rows),
    "Zero sales rows": int(a.zero_sales_rows),
    "Both zero": int(a.both_zero_rows),
    "Exact duplicate excess rows": int(
        key_results["transaction_exact_duplicates"].iloc[0].excess_rows
    ),
    "Conflicting baskets": int(basket_consistency.iloc[0].inconsistent_baskets),
    "Unmatched transaction products": int(
        product_reference.iloc[0].unmatched_transaction_rows
    ),
    "Last observed day": int(h.last_observed_day),
    "Latest full-followup cutoff": int(h.latest_cutoff_with_full_30_day_followup),
}


**Step 23: Inspect the intermediate result.**

In [24]:
display(pd.Series(final_checks, name="Value").to_frame())


,Value
Transaction rows,2595732.00
Baskets,276484.00
Households,2500.00
Households with demographics,801.00
Demographic coverage (%),32.04
Zero quantity rows,14466.00
Zero sales rows,18850.00
Both zero,14399.00
Exact duplicate excess rows,0.00
Conflicting baskets,0.00


## Decisions and remaining questions

1. Preserve all original source rows. No automatic duplicate or anomaly removal has been applied.
2. Build the household dimension from observed transaction identifiers and left join demographics.
3. Keep line, basket, and household counts distinct. Define analytical purchase eligibility before using frequency or average basket value.
4. Standardise whitespace-only product attributes in a later staging layer while retaining the raw source.
5. Validate discount and sales definitions against the original user guide before calculating gross/net revenue or discount rates.
6. Keep relative day numbers until calendar semantics are verified; respect the observed horizon for predictive labels.
7. This is an initial source audit. Remaining work includes coupon and promotion composite keys, redemption relationships, additional semantic missing-value checks, and time-code validation.

The notebook does not estimate campaign uplift, customer lifetime value, or causal incremental profit. Those require additional assumptions and validation.

In [25]:
con.close()
print("Audit notebook complete. Original CSVs were not modified.")


Audit notebook complete. Original CSVs were not modified.
